# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bushrahaji412-lab/MY_ML_INTERNSHIP/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



Which pages in the search dataset show the strongest signals of content-refresh opportunity, and can these signals be used to rank pages for review?

Decision Supported:

This analysis supports the decision of which pages should be prioritized for content review or refresh. The resulting scores will help create a ranked list of pages so that high-priority opportunities can be reviewed first.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



This project uses the FlyRank ML Internship search intelligence dataset. The analysis uses aggregated search and content performance data to identify pages with potential content-refresh opportunities.

The unit of analysis is a content page. Relevant performance signals are aggregated at the page level and used to create features for scoring and ranking.

The analysis excludes private or sensitive information, including client names, domains, URLs, private search queries, credentials, and raw data exports. Only information necessary for the research question is used.

The dataset is used to study patterns and prioritize pages for review. It does not represent Google's ranking algorithm and does not establish causal effects of content changes.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



The analysis follows a page-level content opportunity scoring approach. Features are created from available search and content performance signals and are used to identify pages that may benefit from review or refresh.

The target is defined as a page-level opportunity signal based on observed performance patterns. The model learns the relationship between the selected features and this target, and produces an opportunity score that can be used to rank pages.

The Week 4 baseline score is used as the initial benchmark. The machine learning model is then evaluated against this baseline using the same validation setup.

The data is split into training and testing data so that model performance can be evaluated on observations that were not used during training. Features that could directly reveal the target or use future information are excluded to reduce data leakage.

The goal is not to predict Google's ranking algorithm or prove that a content change will cause a specific outcome. The model is used as a decision-support tool for prioritizing pages for human review.


In [43]:
import pandas as pd
import numpy as np

print("Capstone notebook setup started")

Capstone notebook setup started


In [44]:
# Check what data/files are already available

import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders available:")
print(os.listdir("."))

Current working directory:
/content

Files/folders available:
['.config', 'sample_data']


In [45]:
# Connect to the FlyRank Hugging Face warehouse

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')"
)

BASE = "hf://datasets/FlyRank/internship-warehouse"

print("Connected to FlyRank warehouse!")

Connected to FlyRank warehouse!


In [46]:
# Check available tables/files in the FlyRank warehouse

tables = con.execute(f"""
    SELECT *
    FROM glob('{BASE}/**/*.parquet')
    LIMIT 5
""").fetchdf()

display(tables)

,file
0,hf://datasets/FlyRank/internship-warehouse/dim...
1,hf://datasets/FlyRank/internship-warehouse/dim...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...


In [47]:
# Show the exact available data files

files = con.execute(f"""
    SELECT file
    FROM glob('{BASE}/**/*.parquet')
""").fetchdf()

print(files.to_string(index=False))

                                                                                                  file
                                        hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
                                        hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance

In [48]:
# Check the columns in the search performance data

sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2025-01/data_0.parquet')
    LIMIT 5
""").fetchdf()

print("Columns:")
print(sample.columns.tolist())

display(sample.head())

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [50]:
# Fast page-level sample for capstone
df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions) AS gsc_impressions,
        AVG(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        AVG(ga4_pageviews) AS ga4_pageviews,
        AVG(ga4_sessions) AS ga4_sessions,
        AVG(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2025-01/data_0.parquet')
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
    LIMIT 50000
""").fetchdf()

print("Rows:", len(df))
display(df.head())

Rows: 476


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions
0,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,23.0,0.2,5.207471,0.0,0.0,0.0
1,client_9958f0a7ae1df715,content_c899aef92518c714,8.4,0.2,25.206926,0.0,0.0,0.0
2,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,4.4,0.0,33.230000,0.0,0.0,0.0
3,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,12.8,0.0,14.152625,0.0,0.0,0.0
4,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,28.2,0.0,15.290994,0.0,0.0,0.0


In [51]:
# Create CTR and baseline score

df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

df["impression_rank"] = df["gsc_impressions"].rank(pct=True)
df["low_ctr_rank"] = 1 - df["ctr"].rank(pct=True)

df["baseline_score"] = (
    0.5 * df["impression_rank"] +
    0.5 * df["low_ctr_rank"]
)

print("CTR and baseline score created!")
display(df[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "baseline_score"
]].head(10))

CTR and baseline score created!


,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score
0,content_3b70a18ea133b2bb,23.0,0.2,0.008696,0.509454
1,content_c899aef92518c714,8.4,0.2,0.023810,0.429622
2,content_c7c1d2e68d9d0964,4.4,0.0,0.000000,0.611870
3,content_ae5e5fd6edff550f,12.8,0.0,0.000000,0.709034
4,content_a64143f6e4a21ffe,28.2,0.0,0.000000,0.762080
5,content_e281674658070602,5.0,0.0,0.000000,0.628151
6,content_658f53fa439c66ca,5.4,0.0,0.000000,0.633929
7,content_da9cd3207814ec8d,27.2,0.0,0.000000,0.755252
8,content_96fe7476fada560c,27.8,0.2,0.007194,0.533613
9,content_5ca1b43f9a4d0b01,3.6,0.0,0.000000,0.592437


In [52]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Features
X = df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions"
]]

# Target
y = df["baseline_score"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 380
Testing rows: 96


In [53]:
# Train Decision Tree model

model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model error
model_mae = mean_absolute_error(y_test, y_pred)

print("Decision Tree trained successfully!")
print("Model MAE:", model_mae)

Decision Tree trained successfully!
Model MAE: 0.00499971155663126


In [54]:
# Compare W04 baseline with the same test set

baseline_test = df.loc[X_test.index, "baseline_score"]

baseline_mae = mean_absolute_error(y_test, baseline_test)

print("Baseline MAE:", baseline_mae)
print("Model MAE:", model_mae)

Baseline MAE: 0.0
Model MAE: 0.00499971155663126


In [55]:
# Create a simple opportunity target
# Higher impressions + lower CTR = higher opportunity

df["opportunity_target"] = (
    df["gsc_impressions"].rank(pct=True) +
    (1 - df["ctr"].rank(pct=True))
) / 2

print("Correct target created!")
display(df[[
    "content_hash_id",
    "ctr",
    "baseline_score",
    "opportunity_target"
]].head(10))

Correct target created!


,content_hash_id,ctr,baseline_score,opportunity_target
0,content_3b70a18ea133b2bb,0.008696,0.509454,0.509454
1,content_c899aef92518c714,0.023810,0.429622,0.429622
2,content_c7c1d2e68d9d0964,0.000000,0.611870,0.611870
3,content_ae5e5fd6edff550f,0.000000,0.709034,0.709034
4,content_a64143f6e4a21ffe,0.000000,0.762080,0.762080
5,content_e281674658070602,0.000000,0.628151,0.628151
6,content_658f53fa439c66ca,0.000000,0.633929,0.633929
7,content_da9cd3207814ec8d,0.000000,0.755252,0.755252
8,content_96fe7476fada560c,0.007194,0.533613,0.533613
9,content_5ca1b43f9a4d0b01,0.000000,0.592437,0.592437


In [56]:
# Create an independent outcome target
# Engagement/pageviews represent observed page performance

df["outcome_target"] = (
    df["ga4_pageviews"] +
    df["ga4_sessions"] +
    df["ga4_engaged_sessions"]
)

print("Outcome target created!")
display(df[[
    "content_hash_id",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "outcome_target"
]].head(10))

Outcome target created!


,content_hash_id,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,outcome_target
0,content_3b70a18ea133b2bb,0.0,0.0,0.0,0.0
1,content_c899aef92518c714,0.0,0.0,0.0,0.0
2,content_c7c1d2e68d9d0964,0.0,0.0,0.0,0.0
3,content_ae5e5fd6edff550f,0.0,0.0,0.0,0.0
4,content_a64143f6e4a21ffe,0.0,0.0,0.0,0.0
5,content_e281674658070602,0.0,0.0,0.0,0.0
6,content_658f53fa439c66ca,0.0,0.0,0.0,0.0
7,content_da9cd3207814ec8d,0.0,0.0,0.0,0.0
8,content_96fe7476fada560c,0.0,0.0,0.0,0.0
9,content_5ca1b43f9a4d0b01,0.0,0.0,0.0,0.0


In [57]:
# Check whether GA4 data is usable in this sample

print("Non-zero pageviews:", (df["ga4_pageviews"] > 0).sum())
print("Non-zero sessions:", (df["ga4_sessions"] > 0).sum())
print("Non-zero engaged sessions:", (df["ga4_engaged_sessions"] > 0).sum())
print("Total rows:", len(df))

Non-zero pageviews: 0
Non-zero sessions: 0
Non-zero engaged sessions: 0
Total rows: 476


In [58]:
# Use GSC performance as the outcome
# Higher clicks = stronger observed search outcome

df["outcome_target"] = df["gsc_clicks"]

X = df[[
    "gsc_impressions",
    "gsc_avg_position"
]]

y = df["outcome_target"]

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Features:", X.columns.tolist())
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Features: ['gsc_impressions', 'gsc_avg_position']
Training rows: 380
Testing rows: 96


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



The machine learning model is evaluated against the Week 4 baseline using the same validation setup. The comparison focuses on whether the model provides a more useful ranking of content-refresh opportunities than the baseline approach.

Model performance and baseline performance are reported using the selected evaluation metric. The results are used to determine whether the machine learning approach adds value for prioritizing pages for human review.

The final ranked output is based on the model's opportunity scores. Higher-scoring pages are treated as higher-priority candidates for review or content refresh.

Results are interpreted as decision-support evidence rather than proof of causal impact. The analysis does not claim that refreshing a page will directly improve its search performance.


The machine learning model and the Week 4 baseline were evaluated on the same held-out test split using Mean Absolute Error (MAE).

| Approach               | Evaluation Metric |  Score |
| ---------------------- | ----------------- | -----: |
| Week 4 Baseline        | MAE               | 0.0000 |
| Machine Learning Model | MAE               | 0.0050 |

The Week 4 baseline achieved an MAE of 0.0000, while the Decision Tree model achieved an MAE of 0.0050. Since lower MAE indicates lower prediction error, the Week 4 baseline performed better than the machine learning model in this experiment. Therefore, the current machine learning model does not demonstrate an improvement over the baseline and should be treated as an experimental result rather than a replacement for the baseline.

The results should be interpreted as decision-support evidence and do not prove that a content refresh will cause a specific improvement in search performance.


## 5. Limitations

*What this work cannot claim.*

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



This analysis has several limitations. First, the results are based on observed patterns in the available FlyRank dataset and may not generalize to every website or search environment.

The model identifies pages that appear to have stronger signals of content-refresh opportunity, but it does not prove that refreshing a page will cause higher rankings, clicks, traffic, or engagement.

The analysis cannot claim to reproduce or explain Google's ranking algorithm. It also cannot establish causal relationships between content changes and future search performance.

The resulting scores should therefore be treated as directional decision-support signals rather than guaranteed predictions. Human review is still required before taking content actions.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



The model-generated opportunity scores are used to create a ranked list of pages for content review. Pages with higher scores receive higher priority because they show stronger observed signals associated with content-refresh opportunity.

The recommended action playbook is:

| Priority | Recommendation                     | Suggested Action                                  |
| -------- | ---------------------------------- | ------------------------------------------------- |
| High     | Strong refresh opportunity signals | Review and prioritize for content refresh         |
| Medium   | Some opportunity signals           | Conduct a detailed content and performance review |
| Low      | Limited opportunity signals        | Monitor performance before taking action          |

The ranking is intended to help prioritize limited content-review resources. The highest-ranked pages should be reviewed first, followed by medium-priority pages, while lower-ranked pages can remain under monitoring.

Each recommendation should be reviewed by a human before implementation. The ranking is a decision-support output and should not be interpreted as a guarantee of future search-performance improvement.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


The deployed paper will include the following charts and tables to make the analysis transparent and easy to interpret:

1. **Model vs Baseline Table**
   A table comparing the machine learning model with the Week 4 baseline on the same evaluation split and using the same metric.

2. **Model Performance Chart**
   A visual comparison of the model and baseline performance.

3. **Ranked Opportunity Table**
   A Top-10 table showing the highest-ranked pages, their opportunity scores, reason codes, and recommended actions.

4. **Feature/Signal Summary**
   A chart or table showing the most useful signals used by the model to identify content-refresh opportunities.

5. **Validation Artifact**
   A validation result or diagnostic chart showing how the model performs on held-out data.

All reported values will be generated from the project data and model outputs. No fabricated scores or results will be used.


In [59]:


from sklearn.metrics import mean_absolute_error
import pandas as pd

# 1. Model vs Baseline Table
results_table = pd.DataFrame({
    "Approach": ["Week 4 Baseline", "Machine Learning Model"],
    "Metric": ["MAE", "MAE"],
    "Score": [baseline_mae, model_mae]
})

print("1. Model vs Baseline")
display(results_table)


# 2. Model Performance values
print("2. Model Performance")
print("Baseline MAE:", round(baseline_mae, 4))
print("Model MAE:", round(model_mae, 4))


# 3. Ranked Opportunity Table — Top 10
ranked = df.loc[X_test.index, [
    "content_hash_id",
    "baseline_score"
]].copy()

ranked["model_score"] = y_pred

ranked = ranked.sort_values(
    "model_score",
    ascending=False
).head(10)

ranked["priority"] = "High"
ranked["recommended_action"] = "Review for content refresh"

print("3. Top-10 Ranked Opportunity Table")
display(ranked)


# 4. Feature / Signal Summary
feature_summary = pd.DataFrame({
    "Feature": X.columns
})

print("4. Feature / Signal Summary")
display(feature_summary)


# 5. Validation Artifact
validation = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

validation["Absolute_Error"] = abs(
    validation["Actual"] - validation["Predicted"]
)

print("5. Validation Artifact")
display(validation.head(10))
print("Validation MAE:", round(model_mae, 4))

1. Model vs Baseline


,Approach,Metric,Score
0,Week 4 Baseline,MAE,0.000
1,Machine Learning Model,MAE,0.005


2. Model Performance
Baseline MAE: 0.0
Model MAE: 0.005
3. Top-10 Ranked Opportunity Table


,content_hash_id,baseline_score,model_score,priority,recommended_action
281,content_5e770041ee8f2231,0.771008,0.757283,High,Review for content refresh
249,content_4dbfdffce9a25bc0,0.744223,0.757283,High,Review for content refresh
76,content_a21d564ed74b1065,0.746324,0.757283,High,Review for content refresh
75,content_9151b51c5f4be1c6,0.741597,0.757283,High,Review for content refresh
25,content_c4dca383bdc83897,0.769958,0.757283,High,Review for content refresh
245,content_63c7a1498f76797f,0.740546,0.725315,High,Review for content refresh
15,content_4cfee21dfadec4af,0.735294,0.725315,High,Review for content refresh
455,content_9000d74db4b449c9,0.731092,0.725315,High,Review for content refresh
180,content_e54a2b04d7baea63,0.689076,0.695055,High,Review for content refresh
290,content_cf0500bbb61cc8b9,0.706933,0.695055,High,Review for content refresh


4. Feature / Signal Summary


,Feature
0,gsc_impressions
1,gsc_avg_position


5. Validation Artifact


,Actual,Predicted,Absolute_Error
0,0.0,0.349265,0.349265
1,0.0,0.695055,0.695055
2,0.0,0.349265,0.349265
3,0.0,0.695055,0.695055
4,0.0,0.589519,0.589519
5,0.0,0.436400,0.436400
6,0.0,0.669380,0.669380
7,0.0,0.349265,0.349265
8,0.0,0.456342,0.456342
9,0.0,0.589519,0.589519


Validation MAE: 0.005


In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.